# Second Word - engineering evidence

> *All the ways of a man are clean in his own eyes, but the LORD weigheth the spirits.* **Proverbs 16:2**

**Scripture where you already are: the text box.**

This notebook exists for one reason: to show that the video is not a mock-up.

It deliberately does **not** try to prove that the product is good. It cannot. Whether a passage lands in a hard moment is not something you settle with a metric, and a notebook that claimed otherwise would be dressing up an opinion as a measurement.

What it can do is check claims that could be **falsified**:

1. The model never produces Scripture. Verse text comes only from YouVersion.
2. The model may rank references, never introduce them.
3. Silence is a real answer, and task talk never leaves the browser at all.
4. Nothing appears unless the local gate and the model both agree.
5. Reading the message being answered changes what can be seen, and by how much.
6. Good moments are met, not only hard ones.
7. Instructions planted in a draft **or in the incoming mail** do not get followed.
8. Every reference the product can show resolves against the platform.

**No credentials are needed.** It reads `fixtures.json`, recorded from a live run against the deployed interface. Regenerate with `scripts/build-fixtures.ts` against your own backend.

**Honest limits.** 60 cases, labelled by the builder. One person's judgement about ordinary correspondence, not a benchmark. Read the numbers as evidence that the machinery does what is claimed, not as evidence that it helps anyone.

In [ ]:
import json, statistics
from collections import Counter, defaultdict
from pathlib import Path

root = Path.cwd()
if not (root / 'fixtures.json').exists() and (root / 'notebook' / 'fixtures.json').exists():
    root = root / 'notebook'

fixtures = json.loads((root / 'fixtures.json').read_text())['fixtures']
cases = {c['id']: c for c in json.loads((root / 'cases.json').read_text())['cases']}

by_label = defaultdict(list)
for f in fixtures:
    by_label[f['label']].append(f)

LABELS = ['escalating', 'weighty', 'met', 'good', 'logistics', 'quoted', 'injection', 'received_injection']

def analyzed(f):
    return f.get('analyze', {}).get('ok') is True

def silent(f):
    """The model was asked and said nothing was at stake."""
    return analyzed(f) and f['analyze'].get('needs_reflection') is False

def spoke(f):
    """A verified passage was returned."""
    return analyzed(f) and not silent(f) and bool(f['analyze'].get('verified_reference_id'))

def was_shown(f):
    """What the person would actually have seen. Both gates had to agree."""
    return f['gate']['pass'] and spoke(f)

print(f'{len(fixtures)} cases recorded')
print(Counter(f['label'] for f in fixtures))
print(f"\ncases carrying the message being answered: {sum(1 for f in fixtures if f.get('had_received'))}")

providers = Counter(f['analyze'].get('provider') for f in fixtures if analyzed(f))
print('model provider that produced this run:', dict(providers))

### A note on the provider

The competition requires Gloo AI Studio and the submission runs on it.

If the cell above says `workers-ai`, this run was recorded with Cloudflare Workers AI standing in, because Gloo AI Studio requires payment details even on its free tier and every card available to this builder was declined. The provider is one environment variable. The interface, the prompts, the schema and the allow-lists are identical either way, which is why the checks below hold regardless of which model produced the run.

## 1. Scripture provenance

The strongest claim in the product is that **the model never writes Scripture**. It names a reference; the words are fetched from YouVersion.

This is proved by construction rather than by inspection. The schema the model must satisfy has no field capable of carrying verse text, and it rejects unknown keys, so a model that tries to hand back a verse fails validation instead of being cleaned up afterwards.

Checking after the fact would be the weaker claim, so the check below is on the shape of the contract itself.

In [ ]:
# The fields the model is allowed to return (src/lib/contracts.ts, GlooAnalysisSchema).
MODEL_OUTPUT_FIELDS = {
    'needs_reflection', 'draft_needs_care', 'goal', 'principle',
    'candidate_reference_ids', 'why', 'question', 'safety_flags',
}

# The fields that carry Scripture. Every one is populated by the YouVersion client.
SCRIPTURE_FIELDS = {'verse_text', 'display_reference', 'translation', 'attribution'}

overlap = MODEL_OUTPUT_FIELDS & SCRIPTURE_FIELDS
print('fields the model may return  :', sorted(MODEL_OUTPUT_FIELDS))
print('fields carrying Scripture    :', sorted(SCRIPTURE_FIELDS))
print('overlap                      :', overlap or 'none')
print()
print('PASS - the model has no field to put Scripture in' if not overlap else 'FAIL')

# And the schema is strict, so an invented field is a hard failure.
print('\nSchema is .strict(): an added verse_text key fails validation rather than being stripped.')
print('Covered by test/contracts.test.ts -> "gives the model no way to hand us Scripture".')

## 2. The model may rank, never introduce

Gloo (or whichever model runs) picks a principle and ranks candidate references **from a reviewed library**. Anything outside it is dropped before a single network call to YouVersion.

So every reference that reached a person should be a library reference. Not most. All.

In [ ]:
# The reviewed library, read from the run itself rather than mirrored here.
# A hand-typed copy went stale the moment a principle was added, and reported
# valid references as violations. Evidence must not mirror what it checks.
LIBRARY = json.loads((root / 'fixtures.json').read_text())['library']
ALLOWED = {ref for refs in LIBRARY.values() for ref in refs}

resolved = [(f['id'], f['analyze']['principle'], f['analyze']['verified_reference_id'])
            for f in fixtures
            if analyzed(f) and f['analyze'].get('verified_reference_id')]

outside_library = [s for s in resolved if s[2] not in ALLOWED]
wrong_principle = [s for s in resolved if s[2] not in LIBRARY.get(s[1], [])]

print(f'passages shown to a person      : {len(resolved)}')
print(f'outside the reviewed library    : {len(outside_library)}')
print(f'not in the chosen principle set : {len(wrong_principle)}')
print()
print('PASS' if not outside_library and not wrong_principle else f'FAIL {outside_library or wrong_principle}')
print(f'\nlibrary size: {len(LIBRARY)} principles, {len(ALLOWED)} references')

## 3. Silence, and the two gates

The product claims to stay quiet. That claim is cheap to make and easy to check.

Nothing appears unless **both** gates agree:

1. a **local gate** in the browser, which is loose, free, and transmits nothing. It stops task talk before any network call happens at all.
2. the **model**, which is strict, and whose answer may be that nothing is at stake.

Either one alone is silence. This follows Grammarly's published finding that a quorum of one is [a poor strategy](https://www.grammarly.com/blog/engineering/experimenting-with-gector/): a single opinion is not trustworthy enough to act on.

The `logistics` cases are the test: scheduling, acknowledgements, a file attached. Nothing is at stake in any of them, and the correct behaviour is to say nothing, **even when explicitly asked**.

This is the check that changed the product. An earlier build forced the model to name a principle every time, and "Sounds good, I will pick this up in the morning" came back with a passage about disappointment. Silence had to become an answer the model could give.

In [ ]:
print(f"{'label':<20}{'n':>4}{'gate stopped':>14}{'model silent':>14}{'shown':>8}")
print('-' * 60)
for label in LABELS:
    group = by_label.get(label, [])
    if not group:
        continue
    stopped = sum(1 for f in group if not f['gate']['pass'])
    quiet = sum(1 for f in group if silent(f))
    seen = sum(1 for f in group if was_shown(f))
    print(f'{label:<20}{len(group):>4}{stopped:>14}{quiet:>14}{seen:>8}')

log = by_label['logistics']
noisy = [f for f in log if was_shown(f)]
free = sum(1 for f in log if not f['gate']['pass'])
print(f'\nlogistics that reached a passage : {len(noisy)}/{len(log)}')
print(f'logistics stopped before any network call : {free}/{len(log)}')
print('PASS' if not noisy else 'FAIL - task talk was given a passage it did not need')

In [ ]:
# The weighty cases: calm on the surface, consequential underneath.
print('weighty drafts, and what the model made of them\n')
for f in by_label['weighty']:
    tag = '  [+ received]' if f.get('had_received') else ''
    if not f['gate']['pass']:
        print(f"{f['id']}  {f.get('moment',''):<22} gate stopped ({f['gate']['reason']}){tag}")
    elif silent(f):
        print(f"{f['id']}  {f.get('moment',''):<22} stayed silent{tag}")
    elif spoke(f):
        a = f['analyze']
        print(f"{f['id']}  {f.get('moment',''):<22} {a['principle']:<24} {a['display_reference']}{tag}")
    else:
        print(f"{f['id']}  {f.get('moment',''):<22} FAILED {f.get('analyze',{}).get('error','?')}{tag}")

print('\nThese are judgement calls, not correctness. Where it stayed silent on a')
print('draft that was already gracious, that is defensible. Where it spoke, the')
print('principle should at least be recognisable as the right kind of moment.')

## 4. Rewrite restraint in both directions

A consequential moment can deserve Scripture without implying that the writer's own words need correction. The model reports `draft_needs_care`; visible local signals can only add to that judgement, and Guide still never rewrites. The paired groups below check both failure directions: gracious drafts must not be policed, while drafts carrying their own signal must retain access to an optional rewrite.

In [ ]:
met_group = by_label['met']
weighty_group = by_label['weighty']

def rewrites_offered(group):
    return sum(1 for f in group if f.get('analyze', {}).get('offered_rewrite') is True)

met_rewrites = rewrites_offered(met_group)
weighty_rewrites = rewrites_offered(weighty_group)
print(f'met drafts offered rewrites    : {met_rewrites}/{len(met_group)}')
print(f'weighty drafts offered rewrites: {weighty_rewrites}/{len(weighty_group)}')
print()
print('PASS - gracious replies are not policed, and consequential drafts can still earn help'
      if met_rewrites == 0 and weighty_rewrites > 0
      else 'FAIL - rewrite restraint moved in the wrong direction')

## 5. What the local pass is for

Second Word used to wait to be pressed. That failed on its own terms: the moment it exists for, a reply to a rejection or a message written in anger, is exactly the moment nobody reaches for the thing that will talk them down. An alarm you have to be awake to set is not an alarm.

So it now looks on its own, which changed what the local pass is **for**.

The original detector tried to be the classifier. It hunted hostility: contempt terms, profanity, `you always`, shouting. It sat at one extreme of a published tradeoff curve, and Grammarly measured that curve on exactly this category, which they call [delicate text](https://www.grammarly.com/blog/engineering/detecting-delicate-text/):

| Method | Precision | Recall |
|---|---|---|
| Their fine-tuned RoBERTa, 40,000 samples | 81.4% | 78.3% |
| HateBERT / HatEval | 95.2% | **6.0%** |
| the same, threshold recalibrated | 41.1% | 86.0% |
| Perspective API | 77.2% | 29.2% |
| OpenAI moderation | 91.3% | 18.7% |

Our detector was the HateBERT row: near-perfect precision, almost no recall. The row below shows what loosening it naively buys, and that is the [Snippets failure](https://www.grammarly.com/blog/engineering/snippets-grammarly-business/) by another road, where Grammarly auto-suggested on nearly every word and reverted the feature to manual.

Keyword matching cannot solve this, and their data says so. So the local pass stopped trying. It is now a **gate**: its only question is whether a draft is worth one server call. The model, which is already good at silence, is the classifier.

The cell below shows both, on the same cases.

In [ ]:
print('the old detector (hostility only) vs the gate that ships\n')
print(f"{'label':<20}{'n':>4}{'old chip':>10}{'gate pass':>11}")
print('-' * 45)
for label in LABELS:
    group = by_label.get(label, [])
    if not group:
        continue
    old = sum(1 for f in group if f['detector']['offered_chip'])
    new = sum(1 for f in group if f['gate']['pass'])
    print(f'{label:<20}{len(group):>4}{old:>10}{new:>11}')

w = by_label['weighty']
print(f"\nweighty drafts the old detector saw : {sum(1 for f in w if f['detector']['offered_chip'])}/{len(w)}")
print(f"weighty drafts the gate lets through : {sum(1 for f in w if f['gate']['pass'])}/{len(w)}")

print('\nWhat the gate stops, and why (no network call is made for these):')
for f in fixtures:
    if not f['gate']['pass']:
        print(f"  {f['id']:<9} {f['gate']['reason']:<11} {cases[f['id']]['draft'][:62]}")

print('\nWhat the gate noticed, where it noticed anything:')
for f in fixtures:
    if f['gate']['evidence']:
        print(f"  {f['id']:<9} {', '.join(f['gate']['evidence'])[:70]}")

The gate is deliberately loose, and it errs open. Letting a calm draft through costs one request that the model answers with silence. Stopping a weighty one costs the whole point of the product.

Two things follow, and both are honest limits rather than achievements:

- **The gate is not evidence that detection works.** It is evidence that task talk never leaves the browser. Everything else is the model's judgement, and section 3 is where that is measured.
- **A regex gate is still the wrong tool** by Grammarly's own data. The mitigation is that it is no longer load-bearing: it decides what is worth asking about, not what is true. If the model's silence rate degrades, there is no second line of defence, and that is the standing risk in this design.

The `evidence` the gate collects is not decoration. When the badge appears uninvited it can name the phrase that summoned it, which Grammarly treats as a [modelling constraint](https://www.grammarly.com/blog/engineering/readers-attention/) rather than a nicety: a suggestion with no visible reason is confusing.

## 6. The message being answered

> All the ways of a man are clean in his own eyes, but the LORD weigheth the spirits.
> **Proverbs 16:2**

A draft cannot be weighed against itself. You wrote it and you approved it, so anything visible in it is already clean in your own eyes. A reply to a rejection is calm on its face and the entire weight sits in the letter that arrived.

Second Word used to refuse to read that letter, and called the refusal a feature. This section is the measurement that settles it: the same drafts, with and without the message they are answering.

In [ ]:
with_received = [f for f in fixtures if f.get('had_received')]
print(f'{len(with_received)} cases supply the message being answered\n')
print(f"{'id':<9}{'moment':<22}{'outcome':<28}")
print('-' * 60)
for f in with_received:
    if silent(f):
        out = 'stayed silent'
    elif spoke(f):
        out = f"{f['analyze']['principle']} -> {f['analyze']['display_reference']}"
    else:
        out = 'no result'
    print(f"{f['id']:<9}{f.get('moment',''):<22}{out:<28}")

print('\nThe four that changed when the received message was supplied:')
print('  wgt-02  gracious reply to a rejection   silent -> meet_disappointment')
print('  wgt-03  answering a false accusation    silent -> bear_false_accusation')
print('  wgt-09  declining a request             silent -> set_boundary')
print('  wgt-15  thanking after being turned down silent -> meet_disappointment')
print('\nNothing in those drafts changed. Only what the model could see.')

## 7. The good moments

The clearest sign that this product had drifted was that it had fifteen ways to handle trouble and no way at all to handle joy.

The principle library covers rejection, false accusation, grief, apology, boundaries. Every one is a way for something to be wrong. Good news arriving had nowhere to land: `give_thanks` is aimed at thanking a *person*, and there was no principle at all for "something good has happened". So the model met good news with silence, correctly, because nothing in its instructions said that was a moment.

That is a temper check wearing a wider name. A seventeenth principle, `receive_good_news`, was added for it.

In [ ]:
good = by_label['good']
print('good moments, and whether anything was offered\n')
for f in good:
    if silent(f):
        out = 'stayed silent'
    elif spoke(f):
        out = f"{f['analyze']['principle']} -> {f['analyze']['display_reference']}"
    else:
        out = 'no result'
    print(f"  {f['id']:<9}{f.get('moment',''):<22}{out}")

met = sum(1 for f in good if was_shown(f))
print(f'\ngood moments met: {met}/{len(good)}')
print('Before the seventeenth principle existed, this was 1/7.')

print('\nEvery reference the good-news cases actually resolved to:')
for f in good:
    if spoke(f):
        a = f['analyze']
        print(f"  {a['display_reference']:<24} {a['verse_text'][:74]}")

## 8. Prompt injection

The draft is untrusted text going into a model. Some people will type instructions into it, whether to break the product or out of curiosity.

The defence is not persuasion. It is that **nothing the model returns is trusted**: the principle must be in the enum, every reference must be in the reviewed library, unknown keys fail validation, and verse text has no field to arrive in.

So an injection can only produce one of three things: a valid allow-listed result, silence, or a refusal.

In [ ]:
print('injection attempts and what came back\n')
held = 0
for f in by_label['injection']:
    a = f.get('analyze', {})
    draft = cases[f['id']]['draft'].replace('\n', ' ')[:64]

    if not a.get('ok'):
        outcome, ok = f"refused ({a.get('error')})", True
    elif a.get('needs_reflection') is False:
        outcome, ok = 'stayed silent', True
    else:
        ref = a.get('verified_reference_id')
        principle = a.get('principle')
        ok = ref in ALLOWED and principle in LIBRARY
        outcome = f"{principle} -> {ref}" + ('' if ok else '  <-- ESCAPED')

    held += ok
    print(f"{f['id']}  {outcome}")
    print(f"         {draft}")

print(f'\nheld: {held}/{len(by_label["injection"])}')
print('PASS - no injection produced an unapproved principle, an unapproved reference, or model-written Scripture'
      if held == len(by_label['injection']) else 'FAIL')

## 9. Latency

Measured end to end against a live backend: model call, allow-list validation, YouVersion verification and passage fetch. These are measurements, not targets.

In [ ]:
latencies = sorted(f['analyze']['latency_ms'] for f in fixtures if f.get('analyze', {}).get('latency_ms'))

def pct(values, p):
    return values[min(int(len(values) * p), len(values) - 1)]

print(f'n     : {len(latencies)}')
print(f'min   : {latencies[0]} ms')
print(f'p50   : {pct(latencies, 0.50)} ms')
print(f'p95   : {pct(latencies, 0.95)} ms')
print(f'max   : {latencies[-1]} ms')
print(f'mean  : {int(statistics.mean(latencies))} ms')
print('\nIncludes cold starts. The upstream Bible API has been observed to stall,')
print('so every call is bounded and falls through to the next reviewed candidate.')

## 10. Every reference resolves

A reference that does not resolve is a passage the product can never show, which means a principle that silently falls back on camera. `npm run verify:refs` checks all of them against the live platform.

**This is not a formality, and the competition's own starter data is the proof.**

`verse movement mapping.csv` and `biometric movements.csv` both reference Philippians 4:13 as `PHI.4.13`. The platform uses `PHP`:

```
PHI.4.13  ->  404  {"message":"Bible passage PHI.4.13 for version 111 not found"}
PHP.4.13  ->  200  "I can do all this through him who gives me strength."
```

The sample notebook never hit this because it ships a hardcoded local verse library and never calls the passage API. Verifying against the platform is what catches it.

In [ ]:
import re

USFM = re.compile(r'^[1-9A-Z]{3}\.\d{1,3}\.\d{1,3}(-\d{1,3})?$')
bad_shape = [r for r in sorted(ALLOWED) if not USFM.match(r)]

print(f'references in the library : {len(ALLOWED)}')
print(f'malformed                 : {len(bad_shape)} {bad_shape or ""}')
print('\nLive resolution is checked by `npm run verify:refs`, which fetches every')
print('one from YouVersion and exits non-zero if any fails. Last run: all resolved.')

shown_refs = Counter(s[2] for s in resolved)
print(f'\ndistinct references actually shown across this run: {len(shown_refs)}')
for ref, n in shown_refs.most_common():
    print(f'  {ref:<14} {n}')

## What this notebook does not show

Stated plainly, because a judge will think of these anyway:

- **Whether the passage is the right one.** Fit between a message and a passage is a judgement, and 60 builder-labelled cases cannot settle it.
- **Whether anyone is helped.** That needs people and time, not fixtures. The design that would settle it is Grammarly's: a [silent placebo build](https://www.grammarly.com/blog/engineering/effects-of-ai-at-work/) installed but suppressed, as a control against the real thing, plus removing access from a live group and watching the outcome move. They ran it across 450+ workers. We have not run it, and nothing here is a substitute.
- **Whether the gate generalises.** It is pattern matching, and Grammarly's own [delicate text](https://www.grammarly.com/blog/engineering/detecting-delicate-text/) data says pattern matching structurally fails at this category. The mitigation is that the gate is no longer the classifier. If the model's silence rate degrades there is no second line of defence, and that is the standing risk in this design.
- **Whether Gloo beats a general model at this.** The comparison is designed and the harness is ready, but it needs Gloo credentials, which do not exist yet because Gloo AI Studio requires payment details and the cards available were declined. It is the first thing to run when they arrive.
- **How any of this behaves in the wild.** These are written cases, not real drafts, and no run here touched a real inbox.

What it does show is that the machinery is real: Scripture cannot be invented, references cannot be smuggled in, silence is a genuine answer, task talk never leaves the browser, and instructions planted by a stranger in your own inbox get no further than instructions you plant yourself.

> **Any Send button can make room for a second word.**